# Milestone 6G: End-to-End Big Data AML Architecture Capstone Demo

This capstone notebook orchestrates the complete **Milestone 6 Big Data Enhancement**, executing the integrated pipeline across all six sub-phases:

| Phase | Subsystem | Key Demonstration |
|:---|:---|:---|
| **6A** | Spark RDD Fundamentals | RDD transformations, actions, pair RDDs, exchange rates |
| **6F** | Multi-Source ETL | 3-source ingestion, validation rules, deduplication, enrichment |
| **6E** | Wide-Column Data Model | Delta Lake tables, schema evolution, Cassandra query emulation |
| **6B** | Spark SQL Analytics | Fan-in/out aggregation, hourly velocity windows, mule signatures |
| **6C** | Spark MLlib Classification | Distributed GBT/RF pipelines, PR-AUC evaluation, benchmarks |
| **6D** | Structured Streaming | 5-minute tumbling windows, anomaly rules, real-time alerts |

---


In [1]:
import os
import sys
import asyncio
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))
os.environ['PYTHONPATH'] = os.path.abspath('..') + os.pathsep + os.environ.get('PYTHONPATH', '')

from src.spark_features import _init_windows_hadoop
_init_windows_hadoop()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName('ChainReaction-Capstone')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark Session Ready. Version:', spark.version)


C:\Projects\MULE-T~2\CHAIN-~1\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Session Ready. Version: 4.2.0


## Phase 1: RDD Ingestion & Pair Transformations (Milestone 6A)


In [2]:
from src.spark_rdd_pipeline import run_rdd_pipeline
csv_path = "../data/raw/HI-Small_Trans.csv"
if not os.path.exists(csv_path):
    csv_path = "../data/processed/processed_df.csv"

rdd_results = run_rdd_pipeline(spark.sparkContext, csv_path, sample_fraction=0.01)
print("RDD Phase: Processed", rdd_results["total_accounts"], "accounts.")
print("Riskiest account:", rdd_results["riskiest"][1], "Score:", round(rdd_results["riskiest"][0], 4))


2026-10-29 13:51:55,003 [INFO] src.spark_rdd_pipeline — === Chain Reaction — RDD Feature Pipeline START ===


2026-10-29 13:51:55,083 [INFO] src.spark_rdd_pipeline — Header: Timestamp,From Bank,Account,To Bank,Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering


2026-10-29 13:51:55,086 [INFO] src.spark_rdd_pipeline — Mapped raw lines → Transaction namedtuples (.map).


2026-10-29 13:51:55,175 [INFO] src.spark_rdd_pipeline — Standardised currencies → USD (.map with broadcast).


2026-10-29 13:51:55,176 [INFO] src.spark_rdd_pipeline — Filtered out zero/negative amounts and self-transfers (.filter).


...

2026-10-29 13:52:21,487 [INFO] src.spark_rdd_pipeline — Riskiest account (.first ACTION): 80034A3A0  risk=0.5764


2026-10-29 13:52:25,275 [INFO] src.spark_rdd_pipeline — Label distribution (.countByKey ACTION): {0: 583}


2026-10-29 13:52:29,621 [INFO] src.spark_rdd_pipeline — Saved feature RDD to outputs\rdd_features (.saveAsTextFile ACTION).


2026-10-29 13:52:29,623 [INFO] src.spark_rdd_pipeline — === Chain Reaction — RDD Feature Pipeline END ===


RDD Phase: Processed 583 accounts.
Riskiest account: 80034A3A0 Score: 0.5764


## Phase 2: Multi-Source ETL & Data Quality Validation (Milestone 6F)


In [3]:
from src.etl_pipeline import ETLPipeline
etl = ETLPipeline(
    spark=spark,
    transactions_path=csv_path,
    sanctions_path='../data/external/sanctions_list.csv',
    country_risk_path='../data/external/country_risk_scores.csv',
    max_transaction_rows=25000,
)
etl_result = etl.run()
print('ETL Phase: Quality Audit Summary:')
for k, v in etl.generate_quality_report().items():
    print(f'  {k}: {v}')


2026-10-29 13:52:29,632 [INFO] src.etl_pipeline — Starting Multi-Source ETL Pipeline execution...


2026-10-29 13:52:32,879 [INFO] src.etl_pipeline — Extracted: 25000 transactions, 189 sanctions records, 50 country risk profiles


2026-10-29 13:52:37,691 [INFO] src.etl_pipeline — Validation Complete: 6580 valid, 18420 rejected


C:\Projects\MULE-T~2\CHAIN-~1\Lib\site-packages\pyspark\sql\classic\column.py:361: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


2026-10-29 13:52:42,053 [INFO] src.etl_pipeline — ETL Pipeline completed successfully.


ETL Phase: Quality Audit Summary:
  total_records_by_source: {'transactions': 25000, 'sanctions_watchlists': 189, 'country_risk_ratings': 50}
  records_rejected_by_rule: {'SELF_TRANSFER': 18420}
  total_valid_records: 6580
  deduplication_removed: 0
  sanctions_matches_found: 2
  referential_integrity: {'orphan_senders_count': 0, 'orphan_receivers_count': 0}
  total_accounts_processed: 6751


## Phase 3: Wide-Column Storage & Schema Evolution (Milestone 6E)


In [4]:
from src.wide_column_model import WideColumnStore
store = WideColumnStore(spark, base_storage_dir='../outputs/wide_column_store')
store.create_tables(etl_result.validated_transactions, etl_result.enriched_accounts)
queries = store.simulate_cassandra_queries('8000EBD30')
print('Wide-Column Phase: Materialized View Triage:')
queries['high_risk_cluster_triage'].show(5)


2026-10-29 13:52:42,229 [INFO] src.wide_column_model — Storage engine initialized using format='parquet' at 'C:\Projects\Mule-tiplier Extended\outputs\wide_column_store'


2026-10-29 13:52:42,230 [INFO] src.wide_column_model — Persisting Wide-Column tables...


2026-10-29 13:52:45,078 [INFO] src.wide_column_model — Tables created and registered: aml_transaction_events, aml_account_profiles


2026-10-29 13:52:45,079 [INFO] src.wide_column_model — Executing Cassandra query emulation for account '8000EBD30'...


Wide-Column Phase: Materialized View Triage:


+-------+---------+----------+--------+---------+-------------+
|account|in_degree|out_degree|total_in|total_out|is_laundering|
+-------+---------+----------+--------+---------+-------------+
+-------+---------+----------+--------+---------+-------------+



## Phase 4: Structured Big Data Analytics with Spark SQL (Milestone 6B)


In [5]:
from src.spark_sql_analytics import SparkSQLAnalytics
analytics = SparkSQLAnalytics(spark, etl_result.validated_transactions, etl_result.enriched_accounts)
mules_df = analytics.query_pass_through_mules()
print('Spark SQL Phase: Top Pass-Through Mule Signatures:')
mules_df.show(5)


2026-10-29 13:52:45,569 [INFO] src.spark_sql_analytics — Registered SQL temporary view: 'transactions'


2026-10-29 13:52:45,614 [INFO] src.spark_sql_analytics — Registered SQL temporary view: 'account_features'


Spark SQL Phase: Top Pass-Through Mule Signatures:


+---------+---------+----------+------------+-------------+------------------+---------------+---------------------+--------+------------+
|  account|in_degree|out_degree|total_in_usd|total_out_usd|flow_asymmetry_pct|avg_holding_sec|unique_counterparties|pagerank|ground_truth|
+---------+---------+----------+------------+-------------+------------------+---------------+---------------------+--------+------------+
|8016D6DA0|        3|         4|   134899.46|    116273.13|             14.83|         1800.0|                    2|   0.001|           0|
|8010AFBF0|        2|         5|   125868.88|    130181.16|              3.37|         1800.0|                    2|   0.001|           0|
|800487250|        3|         1|    90907.66|     87038.39|              4.35|         1800.0|                    2|   0.001|           0|
|8024C23E0|        4|         4|    68576.13|     58104.34|             16.53|         1800.0|                    2|   0.001|           0|
|80025B860|        2|      

## Phase 5: Distributed Machine Learning with MLlib (Milestone 6C)


In [6]:
from src.spark_ml_pipeline import SparkMLMuleClassifier
features_path = '../data/processed/account_features.csv'
split_path = '../data/processed/split_accounts.csv'

raw_features = spark.read.option('header', 'true').option('inferSchema', 'true').csv(features_path)
split_df = spark.read.option('header', 'true').csv(split_path)

classifier = SparkMLMuleClassifier(spark)
train_df, val_df, test_df = classifier.prepare_data(raw_features, split_df)

# Fast evaluation on samples
s_train = train_df.sample(False, 0.15, seed=42)
s_val = val_df.sample(False, 0.25, seed=42)
s_test = test_df.sample(False, 0.25, seed=42)

ml_results = classifier.run_benchmark_suite(s_train, s_val, s_test)
classifier.print_comparison_table(ml_results)


2026-10-29 13:52:47,651 [INFO] src.spark_ml_pipeline — Partitioned data using split_accounts mapping.


2026-10-29 13:52:50,639 [INFO] src.spark_ml_pipeline — Splits ready — Train: 360556, Val: 51508, Test: 103016


2026-10-29 13:52:50,648 [INFO] src.spark_ml_pipeline — Training Spark MLlib GBTClassifier...


2026-10-29 13:53:08,540 [INFO] src.spark_ml_pipeline — GBT Complete — Train PR-AUC: 0.1886, Val: 0.0596, Test: 0.0742


2026-10-29 13:53:08,540 [INFO] src.spark_ml_pipeline — Training Spark MLlib RandomForestClassifier...


2026-10-29 13:53:12,518 [INFO] src.spark_ml_pipeline — Random Forest Complete — Test PR-AUC: 0.0616


2026-10-29 13:53:12,519 [INFO] src.spark_ml_pipeline — Training Spark MLlib LogisticRegression...


2026-10-29 13:53:16,931 [INFO] src.spark_ml_pipeline — Logistic Regression Complete — Test PR-AUC: 0.0376



HEAD-TO-HEAD MODEL BENCHMARK COMPARISON TABLE
Model Architecture             | Framework        | Val PR-AUC   | Test PR-AUC 
--------------------------------------------------------------------------------
Random Guessing Floor          | Theoretical      | 0.0123       | 0.0123      
XGBoost (Milestone 2)          | xgboost/sklearn  | —            | 0.0914      
XGBoost (Milestone 3)          | xgboost/sklearn  | 0.0855       | 0.0834      
GraphSAGE GNN (Milestone 3)    | PyTorch Geometric | 0.2091       | 0.1884      
Spark GBT                      | pyspark.ml       | 0.0596       | 0.0742      
Spark Random Forest            | pyspark.ml       | 0.0479       | 0.0616      
Spark Logistic Regression      | pyspark.ml       | 0.0298       | 0.0376      



## Phase 6: Real-Time Structured Streaming (Milestone 6D)


In [7]:
from src.streaming_pipeline import StreamingAnomalyDetector
from src.stream_simulator import simulate_stream

detector = StreamingAnomalyDetector(spark)
simulate_stream(csv_path, detector.incoming_dir, batch_size=100, delay_seconds=0.1, max_batches=2)
queries = detector.start_pipeline(enable_console=True, enable_parquet=False, trigger_interval='1 second', timeout_seconds=5)
print('Streaming Phase Completed successfully.')


2026-10-29 13:53:16,947 [INFO] src.stream_simulator — Reading source dataset from '../data/raw/HI-Small_Trans.csv'...


2026-10-29 13:53:16,996 [INFO] src.stream_simulator — Source loaded: 400 total rows. Emitting micro-batches of 100 rows to 'C:\Projects\Mule-tiplier Extended\notebooks\data\stream\incoming'


2026-10-29 13:53:17,004 [INFO] src.stream_simulator — [13:53:16] Emitted Batch 001: 100 transactions -> txn_stream_1793262196_14c672.csv


2026-10-29 13:53:17,108 [INFO] src.stream_simulator — [13:53:17] Emitted Batch 002: 100 transactions -> txn_stream_1793262197_3d2cbe.csv


2026-10-29 13:53:17,209 [INFO] src.stream_simulator — Reached configured max_batches limit (2). Simulation terminating.


...

2026-10-29 13:53:17,321 [INFO] src.streaming_pipeline — Starting Console Sink query...


2026-10-29 13:53:18,046 [INFO] src.streaming_pipeline — Streaming pipeline launched with 1 active queries.


2026-10-29 13:53:18,047 [INFO] src.streaming_pipeline — Running streaming queries for 5 seconds...


2026-10-29 13:53:23,057 [INFO] src.streaming_pipeline — Timeout elapsed. Stopping queries...


Streaming Phase Completed successfully.


## 7. Comprehensive Requirements Traceability Matrix

| Requirement | Pipeline Deliverable | Verified In Notebook |
|:---|:---|:---|
| **Spark RDD Transformations & Actions** | `src/spark_rdd_pipeline.py` | Cell 2 (RDD Phase) & `06a_spark_rdd_fundamentals.ipynb` |
| **Multi-Source ETL, Mapping & Validation** | `src/etl_pipeline.py` | Cell 3 (ETL Phase) & `06f_etl_pipeline.ipynb` |
| **Wide-Column Data Model & Schema Evolution** | `src/wide_column_model.py`, `docs/cassandra_schema.cql` | Cell 4 (Wide-Column Phase) & `06e_wide_column_model.ipynb` |
| **Spark SQL Structured Analytics** | `src/spark_sql_analytics.py` | Cell 5 (Spark SQL Phase) & `06b_spark_sql_analytics.ipynb` |
| **Spark MLlib Classification & Benchmarking** | `src/spark_ml_pipeline.py` | Cell 6 (MLlib Phase) & `06c_spark_ml_pipeline.ipynb` |
| **Real-Time Structured Streaming** | `src/streaming_pipeline.py`, `src/stream_simulator.py` | Cell 7 (Streaming Phase) & `06d_streaming_pipeline.ipynb` |
| **Databricks Community Edition Readiness** | `notebooks/06*.ipynb` | Micro-cluster compatible execution flow |

---
### Capstone Execution Concluded.


In [8]:
spark.stop()
print('Spark session stopped. Capstone demonstration complete!')


Spark session stopped. Capstone demonstration complete!
